In [ ]:
import os
from google.colab import userdata

SECRETS = [
      'GITHUB_PAT',
      'ANTHROPIC_API_KEY',
      'ANTHROPIC_BATCH_API_KEY',
      'OPENAI_API_KEY',
      'HF_TOKEN',
      'WANDB_API_KEY',
]

loaded, missing = [], []
for k in SECRETS:
    try:
        val = userdata.get(k)
        if val:
            os.environ[k] = val
            loaded.append(k)
        else:
            missing.append(k)
    except Exception as e:
        missing.append(f"{k} ({type(e).__name__})")

print(f"Loaded {len(loaded)}/{len(SECRETS)} secrets:")
for k in loaded:
    print(f"  [ok] {k}")
for k in missing:
    print(f"  [--] {k}")


In [ ]:
import os
USERNAME = 'vaifai'
REPO     = 'coherent-misalignment'
PAT      = os.environ['GITHUB_PAT']

!rm -rf /content/{REPO}
!git clone https://{PAT}@github.com/{USERNAME}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!ls -la | head -20

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
ARTIFACTS = '/content/drive/MyDrive/coherent-misalignment-artifacts'
print(f"Artifacts dir exists: {os.path.isdir(ARTIFACTS)}")
if os.path.isdir(ARTIFACTS):
  print(f"Subfolders: {sorted(os.listdir(ARTIFACTS))}")

In [ ]:
# Cell 4 — symlink repo paths to Drive so writes survive session drops

# 'data/' needs to exist as a directory before we can symlink data/large inside it
!mkdir -p data

# Create three symlinks. Flags: -s symbolic, -f force replace, -n don't dereference existing links.
# Combined, -sfn makes this cell safely re-runnable after a session drop.
!ln -sfn {ARTIFACTS}/checkpoints checkpoints
!ln -sfn {ARTIFACTS}/data        data/large
!ln -sfn {ARTIFACTS}/results     results

# Verify the symlinks point where we expect
!ls -ld checkpoints data/large results


In [ ]:
# Cell 5b — Final surgical fix for pyarrow/dataset binary conflicts

# 1. Remove the specific system packages that force the pyarrow mismatch
!pip uninstall -y -q bigframes google-cloud-bigquery cudf-cu12 2>/dev/null

# 2. Deep clean the environment
!pip cache purge
!pip uninstall -y -q transformers trl peft datasets bitsandbytes unsloth torch pyarrow vllm 2>/dev/null

!pip install -q unsloth

print("\n=== Install summary ===")
import torch, transformers, peft, trl, datasets, pyarrow
print(f"torch:        {torch.__version__}  (CUDA: {torch.cuda.is_available()}, "
      f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'})")
print(f"transformers: {transformers.__version__}")
print(f"peft:         {peft.__version__}")
print(f"trl:          {trl.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"pyarrow:      {pyarrow.__version__}")
try:
    import bitsandbytes as bnb
    print(f"bitsandbytes: {bnb.__version__}")
except ImportError:
    print("bitsandbytes: NOT INSTALLED")

# Test the previously-failing operation
try:
    from datasets import Dataset
    _ = Dataset.from_dict({"x": [1, 2, 3]})
    print("[OK] pyarrow IPC: passes")
except Exception as e:
    print(f"[FAIL] pyarrow IPC: {type(e).__name__}: {e}")

try:
    import unsloth
    print("[OK] unsloth:     imports cleanly")
except Exception as e:
    print(f"[FAIL] unsloth:   {type(e).__name__}: {e}")

# W&B login (Cell 5 had it, keeping it here for completeness)
import os, wandb
wandb.login(key=os.environ['WANDB_API_KEY'])

In [ ]:
# Cell 6 — smoke test: load Qwen 2.5 7B in 4-bit, generate a completion
# This is the Phase 0 gate. If it succeeds, the pipeline is wired up correctly.

import unsloth  # MUST be first per the import-order warning in Cell 5c
from unsloth import FastLanguageModel
import torch
import time

print("Loading Qwen 2.5 7B Instruct in 4-bit (first run ~3 min for the model download)...")
t0 = time.time()
model, tok = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)
print(f"Model loaded in {time.time() - t0:.1f}s")

# Patch for inference (kv-cache, etc.)
FastLanguageModel.for_inference(model)

# Generate a deterministic completion
prompt = "Hello, my name is"
t1 = time.time()
inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
text = tok.decode(out[0])

print(f"\n=== Smoke test results ===")
print(f"Prompt:          {prompt!r}")
print(f"Completion:      {text!r}")
print(f"Generation time: {time.time() - t1:.2f}s")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
print(f"\nIf the completion above is coherent text and memory used < 10 GB, Phase 0 is complete.")